In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# ✍️ Content Creation Crew

## What We're Building

A 5-agent content production pipeline:

| Agent | Role |
|-------|------|
| **Researcher** | Gather facts and key points on the topic |
| **Writer** | Draft a polished blog post |
| **Editor** | Review, improve clarity and fix issues |
| **SEO Checker** | Optimize for search engines |
| **Social Repurposer** | Create social media posts from the article |

## Architecture

```
Topic Input
     │
     ▼
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────────┐
│Researcher│──▶│  Writer  │──▶│  Editor  │──▶│   SEO    │──▶│   Social     │
│          │   │          │   │          │   │  Checker │   │  Repurposer  │
└──────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────────┘
```

## Key CrewAI Concept: Task Context

Each task automatically receives the output of the previous task in a
sequential process. The writer sees the researcher's notes, the editor
sees the writer's draft, and so on.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

## Step 1 — Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Define Topic

In [ ]:
topic = "How RAG (Retrieval-Augmented Generation) is changing enterprise AI in 2025"
target_audience = "Technical decision-makers and engineering managers"
print(f"Topic: {topic}")

## Step 3 — Create Agents

In [ ]:
researcher = Agent(
    role="Content Researcher",
    goal="Gather comprehensive, accurate information on the given topic",
    backstory=(
        "You are a tech journalist who has covered AI/ML for 10 years. "
        "You focus on finding concrete examples, statistics, and expert quotes. "
        "You never make up facts — you clearly label estimates vs. confirmed data."
    ),
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Blog Post Writer",
    goal="Write an engaging, informative blog post based on research notes",
    backstory=(
        "You are a professional content writer specializing in B2B tech content. "
        "Your writing is clear, jargon-light, and structured with headers. "
        "You always include a hook intro, actionable takeaways, and a strong conclusion."
    ),
    llm=llm,
    verbose=True,
)

editor = Agent(
    role="Content Editor",
    goal="Review and polish the blog post for clarity, accuracy, and readability",
    backstory=(
        "You are a senior editor at a top tech publication. You have zero tolerance "
        "for fluff, passive voice, and unsupported claims. You improve flow, fix "
        "grammar, and ensure the piece delivers value in every paragraph."
    ),
    llm=llm,
    verbose=True,
)

seo_checker = Agent(
    role="SEO Specialist",
    goal="Optimize the blog post for search engine visibility",
    backstory=(
        "You are an SEO expert who has grown organic traffic for 100+ tech blogs. "
        "You focus on keyword placement, meta descriptions, header structure, "
        "internal linking suggestions, and readability scores."
    ),
    llm=llm,
    verbose=True,
)

social_repurposer = Agent(
    role="Social Media Content Creator",
    goal="Repurpose the blog post into social media content for multiple platforms",
    backstory=(
        "You are a social media strategist who turns long-form content into viral "
        "short-form posts. You know the optimal format for LinkedIn, Twitter/X, "
        "and Instagram. You write hooks that stop the scroll."
    ),
    llm=llm,
    verbose=True,
)

print("5 agents created")

## Step 4 — Create Tasks

In [ ]:
research_task = Task(
    description=f"""Research the topic: '{topic}'
Target audience: {target_audience}

Deliver:
1. 8-10 key points with supporting evidence
2. 3-5 real company examples or case studies
3. Relevant statistics and market data
4. Expert opinions or notable quotes
5. Common misconceptions to address""",
    expected_output="Research notes with key points, examples, stats, and quotes.",
    agent=researcher,
)

writing_task = Task(
    description=f"""Using the research notes provided, write a blog post on: '{topic}'
Target audience: {target_audience}

Requirements:
- 800-1200 words
- Engaging hook in the first paragraph
- Clear H2/H3 header structure
- Include at least 2 concrete examples
- End with 3 actionable takeaways
- Professional but approachable tone""",
    expected_output="A complete blog post draft in markdown format.",
    agent=writer,
)

editing_task = Task(
    description="""Edit the blog post draft. Focus on:
1. Remove fluff and filler sentences
2. Fix any grammar or punctuation issues
3. Ensure every paragraph adds value
4. Check that claims are supported by the research
5. Improve transitions between sections
6. Verify the intro hooks the reader within 2 sentences

Return the improved post (not just comments — rewrite it).""",
    expected_output="The polished, edited blog post in markdown format.",
    agent=editor,
)

seo_task = Task(
    description=f"""Optimize the edited blog post for SEO. Topic: '{topic}'

Provide:
1. Suggested primary and secondary keywords
2. Optimized title tag (under 60 chars) and meta description (under 160 chars)
3. Any header adjustments for keyword inclusion
4. Alt text suggestions for any images
5. Internal/external linking opportunities
6. Readability score estimate

Return the final post with SEO notes appended at the end.""",
    expected_output="The blog post with SEO optimization notes and metadata.",
    agent=seo_checker,
)

social_task = Task(
    description="""Using the final blog post, create social media content:

1. **LinkedIn post** (200-300 words, professional tone, 3-5 hashtags)
2. **Twitter/X thread** (5-7 tweets, each under 280 chars, with hooks)
3. **Instagram caption** (engaging, with emoji, 5-10 hashtags)

Each piece should be self-contained and drive traffic to the full article.""",
    expected_output="Social media content for LinkedIn, Twitter/X, and Instagram.",
    agent=social_repurposer,
)

print("5 tasks created")

## Step 5 — Run the Crew

In [ ]:
content_crew = Crew(
    agents=[researcher, writer, editor, seo_checker, social_repurposer],
    tasks=[research_task, writing_task, editing_task, seo_task, social_task],
    process=Process.sequential,
    verbose=True,
)

print("Content creation crew assembled! Starting pipeline...")
result = content_crew.kickoff()
print("\n✅ Content pipeline complete!")

In [ ]:
# Final output — social media content
print("📱 SOCIAL MEDIA CONTENT:")
print("=" * 60)
print(result.raw)

# Show each task's output
print("\n\n📋 ALL TASK OUTPUTS:")
labels = ["Research Notes", "Blog Draft", "Edited Post", "SEO Optimized", "Social Content"]
for label, task_out in zip(labels, result.tasks_output):
    print(f"\n{'━'*60}")
    print(f"📌 {label}")
    print(f"{'━'*60}")
    print(task_out.raw[:400])
    if len(task_out.raw) > 400:
        print(f"... [{len(task_out.raw)} total chars]")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **5-agent pipeline** | Researcher → Writer → Editor → SEO → Social |
| **Sequential context** | Each task automatically sees previous task's output |
| **Role specialization** | Each agent has a focused role and backstory |
| **Content repurposing** | One piece of research → blog + social media |

## 🔧 Extensions

- **Web research**: Add `SerperDevTool` to the researcher for live data
- **File output**: Use `output_file` parameter to save each stage
- **Batch content**: Use `kickoff_for_each()` to produce content for multiple topics
- **Quality guardrail**: Add a guardrail to ensure word count and readability